# Exploratory Analysis: Weather Pipeline

This notebook explores the data **produced by the pipeline** (`data/processed/weather_latest.parquet`).
Run `python -m src.pipeline --skip-db` first. Nothing here is hard-coded: every number comes from the data file.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "processed" / "weather_latest.parquet"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run `python -m src.pipeline --skip-db` from the project root first."
    )

df = pd.read_parquet(DATA_PATH)
df["observation_timestamp"] = pd.to_datetime(df["observation_timestamp"])
df["date"] = pd.to_datetime(df["date"])

## 1. Overview

In [ ]:
print("Rows, columns:", df.shape)
print("Period:", df["observation_timestamp"].min(), "->", df["observation_timestamp"].max())
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.groupby("city").size().to_frame("hourly_records"))

## 2. Descriptive statistics

In [ ]:
METRICS = ["temperature_celsius", "apparent_temperature_celsius", "precipitation_mm",
           "wind_speed_kmh", "humidity_pct"]
display(df[METRICS].describe().T)
display(df.groupby("city")[METRICS].agg(["mean", "min", "max"]).round(2))

## 3. Missing values

In [ ]:
missing = df.isna().sum().to_frame("missing_values")
missing["missing_pct"] = (missing["missing_values"] / len(df) * 100).round(2)
display(missing[missing["missing_values"] > 0] if missing["missing_values"].any() else missing.head(0))
print("Total missing cells:", int(df.isna().sum().sum()))

if missing["missing_values"].any():
    missing[missing["missing_values"] > 0]["missing_values"].plot(kind="bar", title="Missing values per column")
    plt.show()

## 4. Temperature analysis

In [ ]:
daily_temp = df.groupby(["date", "city"])["temperature_celsius"].mean().unstack("city")
daily_temp.plot(title="Average daily temperature (°C)")
plt.ylabel("°C")
plt.show()

In [ ]:
hourly_pattern = df.groupby(["hour", "city"])["temperature_celsius"].mean().unstack("city")
hourly_pattern.plot(title="Average temperature by hour of day")
plt.xlabel("Hour")
plt.ylabel("°C")
plt.show()

In [ ]:
df.boxplot(column="temperature_celsius", by="city", rot=20)
plt.title("Temperature distribution by city")
plt.suptitle("")
plt.ylabel("°C")
plt.show()

display(df.groupby("city")["is_extreme_temperature"].sum().to_frame("extreme_temperature_hours"))

## 5. Precipitation analysis

In [ ]:
daily_rain = df.groupby(["date", "city"])["precipitation_mm"].sum().unstack("city")
daily_rain.plot(kind="bar", stacked=False, title="Total daily precipitation (mm)", width=0.8)
plt.ylabel("mm")
plt.tight_layout()
plt.show()

In [ ]:
rain = df.groupby("city").agg(
    total_precipitation_mm=("precipitation_mm", "sum"),
    rainy_hours=("is_rainy", "sum"),
    hours=("is_rainy", "size"),
)
rain["rainy_hours_pct"] = (rain["rainy_hours"] / rain["hours"] * 100).round(2)
display(rain)

## 6. Humidity and wind

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df.groupby(["date", "city"])["humidity_pct"].mean().unstack("city").plot(ax=axes[0], title="Average daily humidity (%)")
df.groupby(["date", "city"])["wind_speed_kmh"].mean().unstack("city").plot(ax=axes[1], title="Average daily wind speed (km/h)")
plt.tight_layout()
plt.show()

## 7. City comparison

In [ ]:
summary = df.groupby("city").agg(
    avg_temperature=("temperature_celsius", "mean"),
    max_temperature=("temperature_celsius", "max"),
    min_temperature=("temperature_celsius", "min"),
    avg_humidity=("humidity_pct", "mean"),
    total_precipitation=("precipitation_mm", "sum"),
    avg_wind_speed=("wind_speed_kmh", "mean"),
).round(2)
display(summary)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
summary["avg_temperature"].plot(kind="bar", ax=axes[0, 0], title="Average temperature (°C)")
summary["total_precipitation"].plot(kind="bar", ax=axes[0, 1], title="Total precipitation (mm)")
summary["avg_humidity"].plot(kind="bar", ax=axes[1, 0], title="Average humidity (%)")
summary["avg_wind_speed"].plot(kind="bar", ax=axes[1, 1], title="Average wind speed (km/h)")
plt.tight_layout()
plt.show()

## 8. Correlations

In [ ]:
corr = df[METRICS].corr().round(2)
display(corr)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
fig.colorbar(image)
ax.set_title("Correlation between metrics")
plt.tight_layout()
plt.show()

## 9. My observations

*Write your own conclusions here after running the notebook with your data
(for example, which city was warmest or rainiest in the period you collected).*